<a href="https://colab.research.google.com/github/ssafy0121/BAEKJOON/blob/master/%ED%81%AC%EB%A1%A4%EB%A7%81(Crawling).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 아래 코드 터미널에 설치하고 시작
# pip install nbformat


import requests
from bs4 import BeautifulSoup
import time
import json
import os
import nbformat

HEADERS = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"}

WORKBOOK_URL = "https://www.acmicpc.net/workbook/view/20440"  # ✅ 바꿀 링크
NOTEBOOK_TITLE = "이름이_긴_문제들_모음"                        # ✅ 바꿀 파일 제목


def get_workbook_problems(workbook_url):
    res = requests.get(workbook_url, headers=HEADERS)
    soup = BeautifulSoup(res.text, "html.parser")
    problems = []
    table = soup.find("table")
    if not table:
        print("문제 테이블을 찾을 수 없습니다.")
        return problems
    for row in table.find_all("tr")[1:]:
        cols = row.find_all("td")
        if len(cols) >= 2:
            problem_id = cols[0].text.strip()
            title_tag = cols[1].find("a")
            title = title_tag.text.strip() if title_tag else cols[1].text.strip()
            problems.append({"id": problem_id, "title": title})
    return problems


def get_problem_detail(problem_id):
    url = f"https://www.acmicpc.net/problem/{problem_id}"
    res = requests.get(url, headers=HEADERS)
    soup = BeautifulSoup(res.text, "html.parser")

    problem_desc = soup.find(id="problem_description")
    input_desc   = soup.find(id="problem_input")
    output_desc  = soup.find(id="problem_output")
    info         = soup.find(id="problem-info")

    samples = []
    i = 1
    while True:
        inp = soup.find(id=f"sample-input-{i}")
        out = soup.find(id=f"sample-output-{i}")
        if not inp and not out:
            break
        samples.append({
            "input":  inp.get_text().strip() if inp else "",
            "output": out.get_text().strip() if out else ""
        })
        i += 1

    source_section = soup.find(id="problem-source")
    source = {}
    if source_section:
        for item in source_section.find_all("li"):
            text = item.get_text().strip()
            if "문제를 만든 사람" in text:
                source["author"] = text.replace("문제를 만든 사람:", "").strip()
            elif "문제의 오타를 찾은 사람" in text:
                source["typo_finders"] = text.replace("문제의 오타를 찾은 사람:", "").strip()
            elif "데이터를 추가한 사람" in text:
                source["data_contributors"] = text.replace("데이터를 추가한 사람:", "").strip()

    return {
        "id":          problem_id,
        "url":         url,
        "description": problem_desc.get_text("\n").strip() if problem_desc else "",
        "input":       input_desc.get_text("\n").strip()   if input_desc  else "",
        "output":      output_desc.get_text("\n").strip()  if output_desc else "",
        "info":        info.get_text(" ").strip()          if info        else "",
        "samples":     samples,
        "source":      source,
    }


def save_as_markdown(problem, folder="problems"):
    os.makedirs(folder, exist_ok=True)
    safe_title = problem.get("title", problem["id"]).replace("/", "_").replace("\\", "_").replace("?", "_").replace("*", "_").replace(":", "_").replace("|", "_").replace("<", "_").replace(">", "_").replace("[", "_").replace("]", "_").replace('"', "_")
    filename = f"{folder}/{problem['id']}_{safe_title}.md"

    samples_md = ""
    for idx, s in enumerate(problem.get("samples", []), 1):
        samples_md += f"\n### 예제 입력 {idx}\n```\n{s['input']}\n```\n### 예제 출력 {idx}\n```\n{s['output']}\n```\n"

    src = problem.get("source", {})
    source_md = ""
    if src:
        source_md = "\n## 📎 출처\n"
        if src.get("author"):            source_md += f"- 문제를 만든 사람: {src['author']}\n"
        if src.get("typo_finders"):      source_md += f"- 오타를 찾은 사람: {src['typo_finders']}\n"
        if src.get("data_contributors"): source_md += f"- 데이터를 추가한 사람: {src['data_contributors']}\n"

    content = f"""# {problem.get('title', problem['id'])} (BOJ {problem['id']})

🔗 {problem['url']}

---

## 📋 문제 정보
{problem['info']}

## 📝 문제
{problem['description']}

## 📥 입력
{problem['input']}

## 📤 출력
{problem['output']}

## 🧪 예제 입출력
{samples_md}
{source_md}
"""
    with open(filename, "w", encoding="utf-8") as f:
        f.write(content)
    print(f"  ✅ md 저장: {filename}")


def save_as_notebook(problems, title):
    nb = nbformat.v4.new_notebook()
    cells = []

    cells.append(nbformat.v4.new_markdown_cell(f"""# 📚 {title}

> 문제집 출처: {WORKBOOK_URL}

각 문제마다 풀이 코드를 아래 빈칸에 작성하세요.
"""))

    for p in problems:
        samples_md = ""
        for idx, s in enumerate(p.get("samples", []), 1):
            samples_md += f"\n**예제 입력 {idx}**\n```\n{s['input']}\n```\n**예제 출력 {idx}**\n```\n{s['output']}\n```\n"

        src = p.get("source", {})
        source_md = f"\n> 출처: {src['author']}" if src.get("author") else ""

        cells.append(nbformat.v4.new_markdown_cell(f"""---

## 🔢 BOJ {p['id']} - {p.get('title', '')}

🔗 {p['url']}
📋 {p['info']}

### 📝 문제
{p['description']}

### 📥 입력
{p['input']}

### 📤 출력
{p['output']}

### 🧪 예제 입출력
{samples_md}
{source_md}
"""))

        cells.append(nbformat.v4.new_code_cell(f"""# BOJ {p['id']} - {p.get('title', '')} 풀이
import sys
input = sys.stdin.readline

def solve():
    pass  # 여기에 풀이 작성

solve()
"""))

    nb.cells = cells
    output_path = f"{title}.ipynb"
    with open(output_path, "w", encoding="utf-8") as f:
        nbformat.write(nb, f)
    print(f"\n  ✅ 노트북 저장: {output_path}")


# ====== 실행 ======
print("📋 문제 목록 가져오는 중...")
problems = get_workbook_problems(WORKBOOK_URL)
print(f"총 {len(problems)}개 문제 발견\n")

all_data = []
for p in problems:
    print(f"🔍 크롤링: {p['id']} - {p['title']}")
    detail = get_problem_detail(p["id"])
    detail["title"] = p["title"]
    all_data.append(detail)
    save_as_markdown(detail)
    time.sleep(1)

with open("problems.json", "w", encoding="utf-8") as f:
    json.dump(all_data, f, ensure_ascii=False, indent=2)

save_as_notebook(all_data, NOTEBOOK_TITLE)

print("\n🎉 완료! problems/ 폴더, problems.json, .ipynb 모두 저장됨")